# minlpkit Quickstart

A tutorial to observe **why it is slow** using `minlpkit` and try **how to fix it** when solving MINLP (Mixed Integer Nonlinear Programming) with PySCIPOpt (SCIP). The flow is:

1. Define a small MINLP on the spot (a batch production plan including the product of integer and continuous variables)
2. Collect observables + diagnose with `mk.analyze`
3. Apply an improvement (exact linearization of the integer-continuous product) with `mk.linearize_product`
4. Compare and verify before/after with `mk.compare_variants`

This notebook allows you to run `import minlpkit` directly within the repository (where the `.venv` is installed in editable mode). To try it externally, refer to the installation cell below.

In [1]:
# インストール(このリポジトリ内では uv sync 済みの環境で実行されるため不要)。
# 単独の環境(Google Colab等)で試す場合は次を実行する:
#
#   %pip install pyscipopt pandas
#   %pip install git+https://github.com/ctenopoma/minlpkit  # 公開後に有効(現在はプレースホルダ)
#
# コアの実行時依存は pyscipopt/pandas/numpy/scipy のみ(ライブモニタ/チューニングのextrasは不要)。


## 1. Model Definition: Batch Production Planning

For each job `j`, determine an integer batch count `n_j` and a continuous batch size `s_j`, ensuring that the effective production `n_j * s_j` satisfies the demand `demand_j`, while minimizing the sum of the setup cost `cost_n_j * n_j` and the size-proportional cost `cost_s_j * s_j`.

The term `n_j * s_j` (a bilinear term of integer × continuous) results in a McCormick relaxation if solved directly by SCIP, which requires spatial branching in the branch-and-bound method. Setting `linearize=True` switches to a version strictly linearized by `mk.linearize_product` (with 0 McCormick gap).

In [2]:
import minlpkit as mk
from pyscipopt import Model, quicksum

JOBS = {
    "J1": dict(demand=12, cost_n=3.0, cost_s=1.0),
    "J2": dict(demand=18, cost_n=2.0, cost_s=1.5),
    "J3": dict(demand=9,  cost_n=4.0, cost_s=0.8),
}
N_MAX = 6
S_MAX = 10.0


def build_model(linearize: bool = False) -> Model:
    """バッチ生産計画。linearize=True で n_j*s_j を厳密線形化した版を返す。"""
    m = Model()
    m.hideOutput()
    n, s, ns = {}, {}, {}
    for j, d in JOBS.items():
        n[j] = m.addVar(vtype="I", lb=1, ub=N_MAX, name=f"n_{j}")
        s[j] = m.addVar(lb=0.0, ub=S_MAX, name=f"s_{j}")
        if linearize:
            ns[j] = mk.linearize_product(m, n[j], s[j], 1, N_MAX, 0.0, S_MAX, f"ns_{j}")
        else:
            ns[j] = n[j] * s[j]  # 双線形のまま(McCormick緩和)
        m.addCons(ns[j] >= d["demand"], name=f"demand_{j}")
    m.setObjective(
        quicksum(JOBS[j]["cost_n"] * n[j] + JOBS[j]["cost_s"] * s[j] for j in JOBS),
        "minimize",
    )
    m.data = dict(n=n, s=s)
    return m


print("baseline (bilinear):", build_model(False))
print("improved (linearized):", build_model(True))


baseline (bilinear): <pyscipopt.scip.Model object at 0x000001917EC1F130>
improved (linearized): <pyscipopt.scip.Model object at 0x000001917EC1F130>


## 2. Observation + Diagnosis (`mk.analyze`)

`analyze` actually solves the model, collects metrics such as dual bound stagnation, spatial branching bias, nonlinear constraint violations, coefficient scales, and symmetry, and then returns symptoms in order of importance based on diagnostic rules.

In [3]:
report = mk.analyze(lambda: build_model(False), name="baseline(bilinear)", time_limit=10)
print(report.summary())
report.metrics


[baseline(bilinear)] 検出症状 0件:


{'gap': 0.0015131339227919646,
 'spatial_share': 0.0,
 'n_stalls': 0,
 'nodes': 1,
 'bottleneck_type': 'demand',
 'bottleneck_rel_viol': 0.49493015597006484,
 'coef_ratio': 12.5,
 'bigm_count': 0,
 'residual_coef_ratio': 12.5,
 'residual_bigm_count': 0,
 'max_linking_groups': 1,
 'n_heavy_linking': 3,
 'n_sym_groups': 0,
 'largest_sym_group': 0,
 'sym_sound': False}

## 3. Apply Improvement

As recommended by the diagnosis, prepare an `improved` model (using `build_model(linearize=True)` from cell 2) where the integer-continuous product `n_j * s_j` is strictly linearized with `mk.linearize_product`. Since the McCormick relaxation gap is eliminated, the dual bound at the root relaxation should be tighter.

In [4]:
baseline = lambda: build_model(linearize=False)
improved = lambda: build_model(linearize=True)


## 4. Before/After Verification (`mk.compare_variants`)

Compare the root dual bound, final gap within the time limit, and number of nodes in a single table.

In [5]:
df = mk.compare_variants({"baseline": baseline, "improved": improved}, time_limit=10)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]


,variant,root_dual,final_dual,final_gap,nodes
0,baseline,37.95,37.892663,0.001513,1
1,improved,37.95,37.950000,0.000000,1


## Summary

- Observe symptoms with `mk.analyze` and apply improvement helpers like `mk.linearize_product` / `mk.pwl_sos2` / `mk.benders` / `mk.column_generation` following the `recipe`.
- Quantitatively compare the root dual bound, gap, and node count before and after improvements with `mk.compare_variants`.
- For larger real-world models (e.g., batch reactor scheduling, Unit Commitment), live visualization of the solving process (`minlpkit.live`), and automatic tuning of SCIP parameters (`minlpkit.tune`), refer to the [User Manual](../docs/manual/index.md) and [Results Gallery](../docs/gallery.md).

> Technical Note: This notebook can also be run on Google Colab (as SCIP is natively bundled in the pyscipopt wheel). It cannot be executed in browser-based Python environments like JupyterLite, as they cannot include SCIP's native binaries.